# 01_cleaning
2023년 소상공인 실태조사 원본 데이터 정제

수행 작업:
1. CP949 인코딩으로 CSV 로딩 및 컬럼명 정상화
2. mapping_table 기준으로 컬럼 선택 및 한글 컬럼명 단축 별칭 부여
3. 종속변수(target) 생성 (사업전환_운영계획코드, 147번)
4. 변수 유형별 결측치 처리 (구조적 결측 vs 무응답)
5. data/processed/clean_data.csv 저장

In [ ]:
import sys, io
if sys.stdout.encoding != 'utf-8':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# 경로 설정 (노트북 기준: src/ 폴더에서 실행)
BASE_DIR = Path().resolve().parent
RAW_PATH = BASE_DIR / "data" / "raw" / "2023_연간자료_등록기반_20260402_03876.csv"
OUT_PATH = BASE_DIR / "data" / "processed" / "clean_data.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR : {BASE_DIR}")
print(f"RAW_PATH : {RAW_PATH}")
print(f"OUT_PATH : {OUT_PATH}")

## 1. 컬럼 별칭 정의
원본 한글명 → 코드에서 사용할 짧은 영문 식별자

In [ ]:
COL_ALIAS = {
    # ── 보조 / 메타 ────────────────────────────
    "조사기준연도":                                     "survey_year",
    "사업체수가중값":                                   "weight",
    "행정구역시도코드":                                 "region_cd",

    # ── 종속변수 ────────────────────────────────
    "사업전환_운영계획코드":                             "target_raw",

    # ── 카테고리 1: 일반 특성 ──────────────────
    "일반_대표자_성별코드":                             "gender",
    "대표자연령대코드":                                 "age_group",
    "일반_소재지_운영장소코드":                         "location_type",
    "일반_합계종사자수":                                "total_workers",
    "일반_사업자형태코드":                              "biz_type",
    "산업대분류코드":                                   "industry_l",
    "산업중분류코드":                                   "industry_m",
    "일반_프랜차이즈가맹점여부":                        "franchise",
    "일반_창업형태코드":                                "startup_type",
    "일반_창업횟수":                                    "startup_count",
    "일반_사업장이전_경험여부":                         "reloc_exp",
    "일반_현사업체_직전종사상지위항목코드":             "prev_status",
    "경영_영업기간_하루평균시간수":                     "op_hours_daily",
    "경영_영업기간_월평균일수":                         "op_days_monthly",
    "경영_영업기간_년간월수":                           "op_months_yearly",
    "경영_사업장면적":                                  "store_area",

    # ── 카테고리 2: 창업 준비 (신규창업자만, 68% 결측) ──
    "창업_동기코드":                                    "startup_motive",
    "창업_준비기간_창업_준비기간_년수":                 "prep_years",
    "창업_준비기간_창업_준비기간_월수":                 "prep_months",
    "창업_준비활동중요성_사업계획서작성코드":           "imp_bizplan",
    "창업_준비활동중요성_시장조사코드":                 "imp_market",
    "창업_준비활동중요성_동종업종종사경험코드":         "imp_exp",
    "창업_준비활동중요성_창업교육코드":                 "imp_edu",
    "창업_준비활동_사업계획서작성여부":                 "did_bizplan",
    "창업_준비활동_시장조사여부":                       "did_market",
    "창업_준비활동_동종업종종사경험여부":               "did_exp",
    "창업_준비활동_창업교육여부":                       "did_edu",
    "창업_어려움정도_입지선정코드":                     "diff_location",
    "창업_어려움정도_업종선택코드":                     "diff_sector",
    "창업_어려움정도_자금조달코드":                     "diff_funding",
    "창업_어려움정도_기술확보코드":                     "diff_tech",
    "창업_어려움정도_인력확보코드":                     "diff_hr",
    "창업_어려움정도_행정절차코드":                     "diff_admin",
    "창업_어려움정도_경영방법코드":                     "diff_mgmt",
    "창업_비용_총창업비용":                             "startup_cost_total",

    # ── 카테고리 3: 경영 성과 ──────────────────
    "경영_매출금액":                                    "sales",
    "경영_영업이익":                                    "op_profit",
    "경영_영업비용":                                    "op_cost",
    "경영_영업비용_급여총액":                           "cost_salary",
    "경영_영업비용_임차료":                             "cost_rent",
    "경영_고객결제방법_현금비율":                       "pay_cash_pct",
    "경영_고객결제방법_카드비율":                       "pay_card_pct",
    "경영_고객결제방법_간편결제비율":                   "pay_easy_pct",
    "경영_점유형태코드":                                "tenure_type",
    "경영_전자상거래_매출실적여부":                     "ecommerce_yn",
    "경영_부채여부":                                    "debt_yn",
    "경영_운영평가_사업장기비전전략보유정도코드":       "eval_vision",
    "경영_운영평가_제품서비스라인변화혁신추구정도코드": "eval_innov",
    "경영_운영평가_위험부담감수도전정도코드":           "eval_risk",
    "경영_운영평가_신제품신기술적용추진정도코드":       "eval_newtech",

    # ── 카테고리 4: 정책 지원 ──────────────────
    "정부지원정책_지원경험코드":                        "policy_exp",
    "정부지원정책_정책자금인지여부":                    "policy_aware",
    "정부지원정책_추진정책1코드":                       "policy_type1",
    "정부지원정책_재난지원1코드":                       "disaster_aid1",

    # ── 카테고리 5: 디지털 전환 (2023 특화) ────
    "경영_운영활동_디지털대응1코드":                   "digi1_kiosk",
    "경영_운영활동_디지털대응2코드":                   "digi2_robot",
    "경영_운영활동_디지털대응3코드":                   "digi3_delivery",
    "경영_운영활동_디지털대응4코드":                   "digi4_sns",
    "경영_운영활동_디지털대응5코드":                   "digi5_erp",
    "경영_운영활동_디지털대응6코드":                   "digi6_qrorder",
    "경영_운영활동_디지털대응7코드":                   "digi7_other",
    "경영_운영활동_디지털대응8코드":                   "digi8_none",
    "경영_운영활동_디지털기술유형도입의향여부":        "digi_intent",
    "정부지원정책_추진정책4코드":                       "policy_digital",
}

GENERAL_COLS = [
    "gender", "age_group", "location_type", "total_workers",
    "biz_type", "industry_l", "industry_m", "franchise",
    "startup_type", "startup_count", "reloc_exp", "prev_status",
    "op_hours_daily", "op_days_monthly", "op_months_yearly", "store_area",
]
STARTUP_COLS = [
    "startup_motive", "prep_years", "prep_months",
    "imp_bizplan", "imp_market", "imp_exp", "imp_edu",
    "did_bizplan", "did_market", "did_exp", "did_edu",
    "diff_location", "diff_sector", "diff_funding",
    "diff_tech", "diff_hr", "diff_admin", "diff_mgmt",
    "startup_cost_total",
]
PERFORMANCE_COLS = [
    "sales", "op_profit", "op_cost", "cost_salary", "cost_rent",
    "pay_cash_pct", "pay_card_pct", "pay_easy_pct",
    "tenure_type", "ecommerce_yn", "debt_yn",
    "eval_vision", "eval_innov", "eval_risk", "eval_newtech",
]
POLICY_COLS = [
    "policy_exp", "policy_aware", "policy_type1", "disaster_aid1",
]
DIGITAL_COLS = [
    "digi1_kiosk", "digi2_robot", "digi3_delivery", "digi4_sns",
    "digi5_erp", "digi6_qrorder", "digi7_other", "digi8_none",
    "digi_intent", "policy_digital",
]
ALL_FEATURE_COLS = GENERAL_COLS + STARTUP_COLS + PERFORMANCE_COLS + POLICY_COLS + DIGITAL_COLS

## 2. 데이터 로딩

In [ ]:
def load_raw(path: Path) -> pd.DataFrame:
    """CP949 인코딩으로 CSV를 읽고 컬럼명을 정상화한다."""
    with open(path, "rb") as f:
        raw_cols = [c.strip().strip(b'"').decode("cp949")
                    for c in f.readline().split(b",")]

    df = pd.read_csv(path, encoding="cp949", header=0, low_memory=False)
    df.columns = raw_cols
    print(f"[로딩] {df.shape[0]:,}행 × {df.shape[1]}열")
    return df


df = load_raw(RAW_PATH)
df.head(3)

## 3. 컬럼 선택 및 별칭 부여

In [ ]:
def rename_and_select(df: pd.DataFrame) -> pd.DataFrame:
    """mapping_table 기준 컬럼만 선택하고 영문 별칭으로 변경."""
    available = {k: v for k, v in COL_ALIAS.items() if k in df.columns}
    missing   = [k for k in COL_ALIAS if k not in df.columns]
    if missing:
        print(f"[경고] 매핑 테이블에 있지만 원본에 없는 컬럼 {len(missing)}개: {missing}")

    df = df[list(available.keys())].copy()
    df = df.rename(columns=available)
    print(f"[컬럼 선택] {df.shape[1]}개 컬럼 유지")
    return df


df = rename_and_select(df)
df.head(3)

## 4. 종속변수 생성
- 코드 1 = 계속운영 → target = 1 (지속)
- 코드 2,3,4 = 업종전환/폐업/은퇴 등 → target = 0 (중단)

In [ ]:
def make_target(df: pd.DataFrame) -> pd.DataFrame:
    df["target"] = (df["target_raw"] == 1).astype(int)

    dist = df["target"].value_counts().sort_index()
    total = len(df)
    print(f"\n[종속변수 분포]")
    print(f"  지속(1): {dist.get(1, 0):,}건 ({dist.get(1, 0)/total*100:.1f}%)")
    print(f"  중단(0): {dist.get(0, 0):,}건 ({dist.get(0, 0)/total*100:.1f}%)")
    print(f"  불균형 비율: {dist.get(1, 0)/max(dist.get(0, 0), 1):.1f}:1\n")
    return df


df = make_target(df)

## 5. 결측치 처리

In [ ]:
def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    변수 유형별 결측치 처리 전략 (brief.md 섹션 4 참조):

    A. 다중응답 미도입 결측 (디지털대응1~7코드)
       → 응답 구조상 '미해당'이 결측으로 표기됨 → 0으로 대체
    B. 조건부 결측 (임차 관련 금액, 자가소유자는 결측)
       → 자가(tenure_type=1)이면 0, 임차자는 -1(불명)로 표시 후 후속 처리
    C. 창업 관련 변수 (신규창업자만 응답, 68% 결측)
       → 인수/승계(startup_type=2) 또는 무응답 → 범주형은 0('미해당'), 수치형은 -1
    D. 전자상거래 매출비율 (89% 결측)
       → 전자상거래 없음(ecommerce_yn=2)이면 0으로 대체
    E. 이진 정책/재난 지원 (결측 = 지원받지 않음)
       → 0으로 대체
    """
    # A: 디지털 다중응답 — 결측 = 미도입(0)
    digi_multi = ["digi1_kiosk", "digi2_robot", "digi3_delivery",
                  "digi4_sns", "digi5_erp", "digi6_qrorder", "digi7_other"]
    for col in digi_multi:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    if "digi8_none" in df.columns:
        has_any_digi = df[digi_multi].sum(axis=1) > 0
        df["digi8_none"] = df["digi8_none"].where(
            df["digi8_none"].notna(),
            other=has_any_digi.map({True: 0, False: np.nan})
        )

    # C: 창업 준비 변수
    startup_cat = [
        "startup_motive",
        "imp_bizplan", "imp_market", "imp_exp", "imp_edu",
        "did_bizplan", "did_market", "did_exp", "did_edu",
        "diff_location", "diff_sector", "diff_funding",
        "diff_tech", "diff_hr", "diff_admin", "diff_mgmt",
    ]
    startup_num = ["prep_years", "prep_months", "startup_cost_total"]

    for col in startup_cat:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    for col in startup_num:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # digi_intent: 결측 = 도입의향 없음(2)
    if "digi_intent" in df.columns:
        df["digi_intent"] = df["digi_intent"].fillna(2)

    # E: 정책/재난 지원 — 결측 = 미수혜(0)
    policy_fill_zero = ["policy_type1", "disaster_aid1", "policy_digital"]
    for col in policy_fill_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 결측률 보고
    remaining = df[ALL_FEATURE_COLS + ["target"]].isnull().sum()
    still_missing = remaining[remaining > 0]
    if len(still_missing):
        print("[결측 잔존 현황]")
        for col, cnt in still_missing.items():
            print(f"  {col}: {cnt:,}건 ({cnt/len(df)*100:.1f}%)")
    else:
        print("[결측치 처리] 모든 feature 컬럼 결측 없음")

    return df


df = handle_missing(df)

## 6. 이진 변수 0/1 정규화
설문에서 ①=1, ②=2로 코딩된 이진 변수를 존재(1) / 부재(0) 체계로 통일

In [ ]:
def normalize_binary(df: pd.DataFrame) -> pd.DataFrame:
    binary_cols = [
        "gender",       # 1=남성, 2=여성 → 1=남성
        "biz_type",     # 1=개인, 2=법인 → 1=개인
        "franchise",    # 1=가맹, 2=독립 → 1=가맹
        "startup_type", # 1=신규, 2=인수 → 1=신규
        "reloc_exp",    # 1=이전경험있음, 2=없음 → 1=있음
        "ecommerce_yn", # 1=전자상거래있음, 2=없음 → 1=있음
        "debt_yn",      # 1=부채있음, 2=없음 → 1=있음
        "policy_exp",   # 1=지원경험있음, 2=없음 → 1=있음
        "policy_aware", # 1=인지함, 2=모름 → 1=인지
        "did_bizplan",  # 1=실시, 2=미실시 → 1=실시
        "did_market",
        "did_exp",
        "did_edu",
        "digi_intent",  # 1=도입의향있음, 2=없음 → 1=있음
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == 1).astype(int)
    return df


df = normalize_binary(df)

## 7. 이상치 Winsorization (연속형 수치 변수)
매출/이익 등 극단값이 많은 연속형 변수를 상하 1% 기준으로 클리핑

In [ ]:
def winsorize(df: pd.DataFrame, lower: float = 0.01, upper: float = 0.99) -> pd.DataFrame:
    continuous_cols = [
        "total_workers", "startup_count", "store_area",
        "op_hours_daily", "op_days_monthly", "op_months_yearly",
        "prep_years", "prep_months", "startup_cost_total",
        "sales", "op_profit", "op_cost", "cost_salary", "cost_rent",
        "pay_cash_pct", "pay_card_pct", "pay_easy_pct",
    ]
    for col in continuous_cols:
        if col not in df.columns:
            continue
        lo = df[col].quantile(lower)
        hi = df[col].quantile(upper)
        df[col] = df[col].clip(lo, hi)
    return df


df = winsorize(df)

## 8. 최종 컬럼 선택 및 저장

In [ ]:
final_cols = ["survey_year", "weight"] + ALL_FEATURE_COLS + ["target_raw", "target"]
final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols].reset_index(drop=True)

df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[저장 완료] {OUT_PATH}")
print(f"  최종 shape: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"  target=1 (지속): {df['target'].sum():,}건")
print(f"  target=0 (중단): {(df['target'] == 0).sum():,}건")

df.head()